# Solving TSP with QAOA

## Test examples :

In [1]:
# W = [
#     [0, 10, 1, 3],
#     [10, 0, 4, 2],
#     [1, 4, 0, 2],
#     [3, 2, 2, 0]
# ]

# W = [
#     [0, 5, 1],
#     [5, 0, 4],
#     [1, 4, 0],
# ]

# n = 2

# W = [
#     [0, 2],
#     [2, 0]
# ]

W = [
    [  0.00,  26.57,  12.07,  47.19, 126.81,  35.69,  77.42,  76.09, 125.47,  78.93],
    [ 40.15,   0.00,  41.21,  78.54, 118.78,  41.20,  91.83,  85.67, 134.36, 140.95],
    [ 12.07,  70.52,   0.00,  39.15,  64.20,  92.26, 115.77, 117.75, 141.45,  39.78],
    [ 62.83,  40.30,  39.15,   0.00,  48.47,  15.35,  96.51,  49.51,  90.94,   0.77],
    [172.35, 134.38,  64.20,  27.94,   0.00,  48.81,  26.91,  23.45,  31.52,  28.75],
    [ 90.54,  41.20,  42.21,   3.04,  49.72,   0.00,  68.40,  48.17, 117.34,   8.62],
    [ 76.64,  89.36,  87.14,  52.94,  26.91,  94.27,   0.00,  12.55,   8.96,  67.32],
    [ 77.27,  85.52,  87.38,  52.82,  23.46,  48.17,   6.24,   0.00,   9.16,  67.09],
    [130.99, 133.05, 110.34,  73.83,  31.52,  83.61,  11.19,   8.96,   0.00,  58.48],
    [113.36,  74.74,  40.17,   0.77,  38.26,   8.56, 134.50,  65.68,  58.58,   0.00],
]

W = W[:4][:4]


n = len(W)

print(f"Number of cities : {n}")
print(f"Number of variables : {n**2}")
print(f"Size of hamiltonian : {2**(n**2)}")



Number of cities : 4
Number of variables : 16
Size of hamiltonian : 65536


## QP Formulation

In [2]:
import qiskit
from qiskit_algorithms import QAOA
from qiskit_optimization import QuadraticProgram

In [3]:

qp = QuadraticProgram(name="TSP")

# Variables: x[i][p] = 1 if city i is at position p 
for i in range(n):
    for p in range(n):
        qp.binary_var(name=f"x_{i}_{p}")

def x(i, p): return f"x_{i}_{p}"

# Objective: minimize total travel distance 
# sum over all edges (i,j) and positions p of dist[i][j] * x[i][p] * x[j][(p+1)%n]
quadratic_terms = {}
for i in range(n):
    for j in range(n):
        if i != j:
            for p in range(n - 1):
                q = p + 1
                key = (x(i,p), x(j,q))
                quadratic_terms[key] = quadratic_terms.get(key, 0) + W[i][j]

qp.minimize(quadratic=quadratic_terms)

# Constraint 1: each city visited exactly once 
for i in range(n):
    qp.linear_constraint(
        linear={x(i,p): 1 for p in range(n)},
        sense="==",
        rhs=1,
        name=f"city_{i}_once"
    )

# Constraint 2: each position has exactly one city 
for p in range(n):
    qp.linear_constraint(
        linear={x(i,p): 1 for i in range(n)},
        sense="==",
        rhs=1,
        name=f"position_{p}_once"
    )

print(qp.export_as_lp_string())

\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: TSP

Minimize
 obj: [ 53.140000000000 x_0_0*x_1_1 + 24.140000000000 x_0_0*x_2_1
      + 94.380000000000 x_0_0*x_3_1 + 80.300000000000 x_0_1*x_1_0
      + 53.140000000000 x_0_1*x_1_2 + 24.140000000000 x_0_1*x_2_0
      + 24.140000000000 x_0_1*x_2_2 + 125.660000000000 x_0_1*x_3_0
      + 94.380000000000 x_0_1*x_3_2 + 80.300000000000 x_0_2*x_1_1
      + 53.140000000000 x_0_2*x_1_3 + 24.140000000000 x_0_2*x_2_1
      + 24.140000000000 x_0_2*x_2_3 + 125.660000000000 x_0_2*x_3_1
      + 94.380000000000 x_0_2*x_3_3 + 80.300000000000 x_0_3*x_1_2
      + 24.140000000000 x_0_3*x_2_2 + 125.660000000000 x_0_3*x_3_2
      + 82.420000000000 x_1_0*x_2_1 + 157.080000000000 x_1_0*x_3_1
      + 141.040000000000 x_1_1*x_2_0 + 82.420000000000 x_1_1*x_2_2
      + 80.600000000000 x_1_1*x_3_0 + 157.080000000000 x_1_1*x_3_2
      + 141.040000000000 x_1_2*x_2_1 + 82.420000000000 x_1_2*x_2_3
      + 80.600000000000 x_1_2*x_3_1 + 157.

## Convert to QUBO

In [4]:
from qiskit_optimization.converters import (
    QuadraticProgramToQubo,   # folds constraints into objective via penalties
    LinearEqualityToPenalty,  # sub-step: converts == constraints to penalty terms
    MinimizeToMaximize,       # flips sign if needed
    IntegerToBinary,          # converts integer vars to binary (not needed here)
)
from qiskit_optimization.translators import to_ising
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
import numpy as np

# Step 1: QP → QUBO  
qubo_converter = QuadraticProgramToQubo()
qubo = qubo_converter.convert(qp)
print(qubo)

# Step 2: QUBO → Ising Hamiltonian 
ising_operator, offset = to_ising(qubo)


minimize 3060.5000000000005*x_0_0^2 + 3060.5000000000005*x_0_0*x_0_1 + 3060.5000000000005*x_0_0*x_0_2 + 3060.5000000000005*x_0_0*x_0_3 + 3060.5000000000005*x_0_0*x_1_0 + 26.57*x_0_0*x_1_1 + 3060.5000000000005*x_0_0*x_2_0 + 12.07*x_0_0*x_2_1 + 3060.5000000000005*x_0_0*x_3_0 + 47.19*x_0_0*x_3_1 + 3060.5000000000005*x_0_1^2 + 3060.5000000000005*x_0_1*x_0_2 + 3060.5000000000005*x_0_1*x_0_3 + 40.15*x_0_1*x_1_0 + 3060.5000000000005*x_0_1*x_1_1 + 26.57*x_0_1*x_1_2 + 12.07*x_0_1*x_2_0 + 3060.5000000000005*x_0_1*x_2_1 + 12.07*x_0_1*x_2_2 + 62.83*x_0_1*x_3_0 + 3060.5000000000005*x_0_1*x_3_1 + 47.19*x_0_1*x_3_2 + 3060.5000000000005*x_0_2^2 + 3060.5000000000005*x_0_2*x_0_3 + 40.15*x_0_2*x_1_1 + 3060.5000000000005*x_0_2*x_1_2 + 26.57*x_0_2*x_1_3 + 12.07*x_0_2*x_2_1 + 3060.5000000000005*x_0_2*x_2_2 + 12.07*x_0_2*x_2_3 + 62.83*x_0_2*x_3_1 + 3060.5000000000005*x_0_2*x_3_2 + 47.19*x_0_2*x_3_3 + 3060.5000000000005*x_0_3^2 + 40.15*x_0_3*x_1_2 + 3060.5000000000005*x_0_3*x_1_3 + 12.07*x_0_3*x_2_2 + 3060.50

In [5]:
from qiskit_aer.primitives import Sampler as AerSampler

sampler = AerSampler(
    backend_options={
        "method": "statevector", #matrix_product_state
        "max_parallel_threads": 0,      # 0 = all cores
        "max_parallel_experiments": 0,
        "max_parallel_shots": 0
    },
    run_options={
        "shots": 1024
    }
)

optimizer = COBYLA()

## Solve QAOA

In [6]:

# Step 3: feed to QAOA
qaoa = QAOA(
    sampler=sampler,
    optimizer=optimizer,
    reps=5,               # p layers — more layers = better approximation, more circuit depth
    initial_point=None,   # optional initial [β, γ] parameters, length = 2p
    mixer=None,           # default X mixer; can provide custom circuit for constrained problems
)
result = qaoa.compute_minimum_eigenvalue(ising_operator)
print(result)


# Step 4: interpret result back in terms of original variables
bitstring = result.best_measurement["bitstring"]           # e.g. "1000010000100001"
x_array   = np.array([int(b) for b in bitstring])         # → [1,0,0,0,0,1,0,0,...]

decoded = qubo_converter.interpret(x_array)
print(decoded)

{   'aux_operators_evaluated': None,
    'best_measurement': {   'bitstring': '1000010000010010',
                            'probability': 0.0009765625,
                            'state': 33810,
                            'value': np.complex128(-24774.942499999997+0j)},
    'cost_function_evals': np.int64(84),
    'eigenstate': {34326: 0.0009765625, 60029: 0.0009765625, 48267: 0.0009765625, 9183: 0.0009765625, 48374: 0.0009765625, 32897: 0.0009765625, 4221: 0.0009765625, 37033: 0.0009765625, 53869: 0.0009765625, 44833: 0.0009765625, 27172: 0.0009765625, 14799: 0.0009765625, 17005: 0.0009765625, 55951: 0.0009765625, 20694: 0.0009765625, 6945: 0.0009765625, 4470: 0.0009765625, 32645: 0.0009765625, 7118: 0.0009765625, 48523: 0.0009765625, 23771: 0.0009765625, 63699: 0.0009765625, 37920: 0.0009765625, 27700: 0.0009765625, 32824: 0.0009765625, 10645: 0.0009765625, 40075: 0.0009765625, 19356: 0.0009765625, 33207: 0.0009765625, 47530: 0.0009765625, 24423: 0.0009765625, 23942: 0.000976562

## Convert to readable path

In [7]:
import numpy as np

def from_bstring_to_circuit(bstring) :
    bstring = bstring[::-1]
    buf = []
    for i in range(n) :
        for j in range(n) :
            if bstring[i*n + j] == 1 :
                buf.append(j)
    order = np.argsort(buf)
    
    out_str = f""
    for i in range(n - 1) :
        out_str += f" {order[i]} ->"
    out_str += f" {order[n - 1]}"
    print(out_str)
    return order

from_bstring_to_circuit(decoded)

 1 -> 0 -> 2 -> 3


array([1, 0, 2, 3])